# FastEnhancer 手作課：讓聲音更清晰
# FastEnhancer Hands-on: Making Speech Clearer

這份 notebook 帶你一步一步完成**串流語音降噪**，從安裝到降噪都只要依序按下每個 cell 左邊的 ▶ 執行鍵，不需要改任何程式。

🇬🇧 This notebook walks you through **streaming speech enhancement** step by step. Just press the ▶ button on the left of each cell, in order. No coding needed.

- 模型 Model：[FastEnhancer](https://github.com/aask1357/fastenhancer)（Ahn et al., 2025），以 ONNX Runtime 執行 / run with ONNX Runtime
- 只需要 CPU：Colab 選單「執行階段 → 變更執行階段類型」選 **CPU** 就夠了
  🇬🇧 CPU is enough: *Runtime → Change runtime type → CPU*
- 想在手機上即時試玩，請打開降噪網頁版（網址見最後一節）
  🇬🇧 To try live denoising on your phone, open the web version (link in the last section)

| 步驟 Step | 內容 | Content |
|---|---|---|
| 0 | 安裝與下載 | Setup and downloads |
| 1 | 聽聽看、看頻譜圖 | Listen and look at spectrograms |
| 2 | 認識模型：輸入、輸出、記憶 | The model: input, output, memory |
| 3 | 串流降噪：一次 16 毫秒 | Streaming: 16 ms at a time |
| 4 | 速度：模型跑得比聲音快嗎？ | Speed: faster than real time? |
| 5 | 延遲：聲音晚了多少？ | Latency: how late is the output? |
| 6 | 實驗：拿掉記憶會怎樣？ | Experiment: what if we remove the memory? |
| 7 | 實驗：噪音越大，模型還撐得住嗎？ | Experiment: how much noise can it handle? |
| 8 | 錄你自己的聲音 | Record your own voice |

## 0. 安裝與下載 / Setup and downloads

安裝 `onnxruntime`（執行模型的引擎），並從 FastEnhancer 官方 GitHub 下載三個大小的模型：Tiny、Base、Small。
這三個模型都用 DNS-Challenge 的大量真實噪音訓練，取樣率 16 kHz。

🇬🇧 Install `onnxruntime` (the engine that runs the model) and download three model sizes from the official FastEnhancer GitHub: Tiny, Base and Small.
All three were trained on the DNS-Challenge data with lots of real-world noise, at a 16 kHz sampling rate.

In [ ]:
!pip -q install onnxruntime soundfile

import os, time, urllib.request
import numpy as np
import soundfile as sf
import matplotlib.pyplot as plt
import onnxruntime as ort
from IPython.display import Audio, display

RELEASE = "https://github.com/aask1357/fastenhancer/releases/download/onnx-dns-v1.0.0"
# 降噪網頁版（範例音檔放在這裡）/ the web version (example clips live here)
SITE = os.environ.get("FE_SITE", "https://benevolent-pithivier-3ffb14.netlify.app")

os.makedirs("models", exist_ok=True)
os.makedirs("samples", exist_ok=True)
for size in ["t", "b", "s"]:
    path = f"models/fastenhancer_{size}.onnx"
    if not os.path.exists(path):
        try:
            urllib.request.urlretrieve(f"{RELEASE}/fastenhancer_{size}.onnx", path)
        except Exception:   # 連不到 GitHub 時，改從降噪網頁版下載 / if GitHub is blocked, use the web version's copy
            urllib.request.urlretrieve(f"{SITE}/models/dns16k_{size}.onnx", path)
    print(path, f"{os.path.getsize(path) / 1024:.0f} KB")

for name in ["cafe", "metro", "station", "kitchen"]:
    for kind in ["noisy", "clean"]:
        path = f"samples/{name}_{kind}.wav"
        if not os.path.exists(path):
            urllib.request.urlretrieve(f"{SITE}/samples/{name}_{kind}.wav", path)
print("下載完成 / Download complete")

## 1. 聽聽看、看頻譜圖 / Listen and look at spectrograms

範例是英文語音混入真實環境噪音（咖啡廳、捷運車廂、車站、廚房）。
**SNR**（訊號雜訊比）表示人聲比噪音大多少，0 dB 代表兩者一樣大聲。

**頻譜圖**把聲音畫成一張圖：橫軸是時間，縱軸是頻率（下面低音、上面高音），越亮代表那個頻率越大聲。
人聲會呈現一條條橫向的「條紋」（諧波），噪音則像一片霧。

🇬🇧 The examples are English speech mixed with real-world noise (café, metro car, station, kitchen).
**SNR** (signal-to-noise ratio) says how much louder the speech is than the noise; 0 dB means equally loud.

A **spectrogram** draws sound as a picture: time runs left to right, frequency bottom to top (low to high pitch), and brighter means louder.
Speech shows up as horizontal *stripes* (harmonics); noise looks like a fog.

In [ ]:
SR = 16000

def load(path):
    x, sr = sf.read(path, dtype="float32")
    assert sr == SR
    return x

def spec(ax, x, title):
    x = x + 1e-6 * np.random.randn(len(x))   # 避免完全無聲的地方算出 -∞ / avoid -inf on pure silence
    ax.specgram(x, NFFT=512, Fs=SR, noverlap=384, cmap="magma", vmin=-110, vmax=-20)
    ax.set_title(title)
    ax.set_ylabel("頻率 Frequency (Hz)")

# 讓 matplotlib 能顯示中文：下載思源黑體 / download a font that can show Chinese in the plots
import warnings, matplotlib.font_manager as fm
warnings.filterwarnings("ignore")
FONT = "NotoSansTC.ttf"
if not os.path.exists(FONT):
    urllib.request.urlretrieve("https://github.com/google/fonts/raw/main/ofl/notosanstc/NotoSansTC%5Bwght%5D.ttf", FONT)
fm.fontManager.addfont(FONT)
plt.rcParams["font.family"] = [fm.FontProperties(fname=FONT).get_name(), "DejaVu Sans"]
plt.rcParams["axes.unicode_minus"] = False

noisy = load("samples/cafe_noisy.wav")
clean = load("samples/cafe_clean.wav")
print(f"長度 Length: {len(noisy) / SR:.1f} 秒 s，共 {len(noisy)} 個取樣點 samples")

fig, axes = plt.subplots(2, 1, figsize=(10, 5), sharex=True)
spec(axes[0], noisy, "有噪音 Noisy（咖啡廳 café，SNR 0 dB）")
spec(axes[1], clean, "乾淨答案 Clean reference")
axes[1].set_xlabel("時間 Time (s)")
plt.tight_layout(); plt.show()

print("有噪音 Noisy:"); display(Audio(noisy, rate=SR))
print("乾淨答案 Clean:"); display(Audio(clean, rate=SR))

## 2. 認識模型：輸入、輸出、記憶 / The model: input, output, memory

先看看模型要吃什麼、吐什麼：

- `wav_in`：**一小段**聲音，256 個取樣點。16 kHz 下等於 256 / 16000 = **16 毫秒**
- `cache_in_*`：模型的**記憶**。它只看得到這 16 毫秒，需要記住前面聽過什麼，才知道哪些是持續的噪音
- 輸出 `wav_out` 是降噪後的 16 毫秒，`cache_out_*` 是更新後的記憶，下一輪要再餵回去

🇬🇧 What goes in and what comes out:

- `wav_in`: a **short piece** of audio, 256 samples. At 16 kHz that is 256 / 16000 = **16 ms**
- `cache_in_*`: the model's **memory**. It only sees these 16 ms, so it must remember what it heard before to tell which sounds are ongoing noise
- The output `wav_out` is the denoised 16 ms; `cache_out_*` is the updated memory, fed back in on the next step

In [ ]:
def open_model(size="b"):
    opts = ort.SessionOptions()
    opts.intra_op_num_threads = 1   # 只用一個 CPU 核心，模擬手機或筆電 / one CPU core, like a phone or laptop
    return ort.InferenceSession(f"models/fastenhancer_{size}.onnx", opts, providers=["CPUExecutionProvider"])

model = open_model("b")
print("輸入 Inputs:")
for i in model.get_inputs():
    print(f"  {i.name:12s} 形狀 shape {i.shape}")
print("輸出 Outputs:")
for o in model.get_outputs():
    print(f"  {o.name:12s} 形狀 shape {o.shape}")

HOP = model.get_inputs()[0].shape[1]
print(f"\n一次處理 {HOP} 個取樣點 = {HOP / SR * 1000:.0f} 毫秒 / {HOP} samples per step = {HOP / SR * 1000:.0f} ms")

## 3. 串流降噪：一次 16 毫秒 / Streaming: 16 ms at a time

這是整堂課的核心。真正的即時系統（視訊會議、助聽器）拿不到整段錄音，只能**一邊收音、一邊處理**。
所以我們把聲音切成一格一格的 16 毫秒，依序餵給模型，並且每次都把記憶傳下去。

🇬🇧 This is the heart of the course. A real-time system (video calls, hearing aids) never has the whole recording; it must **process while it listens**.
So we cut the sound into 16 ms pieces, feed them to the model one by one, and pass the memory along each time.

```
聲音 Audio:  [16ms][16ms][16ms][16ms] ...
                ↓     ↓     ↓     ↓
模型 Model:   處理 → 處理 → 處理 → 處理 ...   （→ = 記憶 memory / cache 往下傳 passed on）
                ↓     ↓     ↓     ↓
輸出 Output: [16ms][16ms][16ms][16ms] ...
```

In [ ]:
def enhance_streaming(model, x, keep_memory=True):
    hop = model.get_inputs()[0].shape[1]
    n = len(x)
    # 記憶一開始是全部 0 / the memory starts as all zeros
    memory = {i.name: np.zeros(i.shape, np.float32) for i in model.get_inputs() if i.name.startswith("cache_in")}
    # 補 0 湊成整數格，最後再多 1 格，把延遲的尾巴推出來 / pad to whole steps, plus one step to flush the delay
    x = np.concatenate([x, np.zeros((-n) % hop + hop, np.float32)])
    out = []
    for start in range(0, len(x), hop):
        chunk = x[start:start + hop][None, :]              # 取出 16 毫秒 / take 16 ms
        results = model.run(None, {"wav_in": chunk, **memory})
        out.append(results[0][0])                           # 降噪後的 16 毫秒 / denoised 16 ms
        if keep_memory:                                     # 把新的記憶傳給下一格 / pass the new memory on
            for j, name in enumerate(memory):
                memory[name] = results[j + 1]
    y = np.concatenate(out)
    return y[hop:hop + n]                                   # 去掉 1 格延遲 / remove the one-step delay

enhanced = enhance_streaming(model, noisy)

fig, axes = plt.subplots(3, 1, figsize=(10, 7), sharex=True)
spec(axes[0], noisy, "有噪音 Noisy")
spec(axes[1], enhanced, "FastEnhancer 降噪後 Denoised")
spec(axes[2], clean, "乾淨答案 Clean reference")
axes[2].set_xlabel("時間 Time (s)")
plt.tight_layout(); plt.show()

print("降噪後 Denoised:"); display(Audio(enhanced, rate=SR))

**換別的範例試試**：把下面的 `"metro"` 換成 `"station"` 或 `"kitchen"` 再執行。
注意廚房的鍋碗碰撞聲：這種突然出現的噪音，比咖啡廳那種持續的嗡嗡聲難處理。

🇬🇧 **Try another example**: change `"metro"` below to `"station"` or `"kitchen"` and run again.
Listen to the clattering in the kitchen: sudden noises like that are harder than the steady hum of a café.

In [ ]:
name = "metro"   # 可換成 / try: "cafe", "station", "kitchen"

x = load(f"samples/{name}_noisy.wav")
y = enhance_streaming(model, x)
fig, axes = plt.subplots(2, 1, figsize=(10, 4.5), sharex=True)
spec(axes[0], x, f"{name}：有噪音 Noisy")
spec(axes[1], y, f"{name}：降噪後 Denoised")
plt.tight_layout(); plt.show()
display(Audio(x, rate=SR)); display(Audio(y, rate=SR))

## 4. 速度：模型跑得比聲音快嗎？ / Speed: faster than real time?

即時處理的條件是：處理 16 毫秒的聲音，必須花**少於 16 毫秒**。

**RTF**（Real-Time Factor，即時因子）= 運算時間 ÷ 聲音長度。
- RTF < 1：跟得上即時
- RTF = 0.1：處理 1 秒的聲音只要 0.1 秒

下面比較 Tiny、Base、Small 三個模型。都只用**一個 CPU 核心**，跟手機或筆電差不多。

🇬🇧 To run in real time, processing 16 ms of audio must take **less than 16 ms**.

**RTF** (real-time factor) = compute time ÷ audio length.
- RTF < 1: keeps up with real time
- RTF = 0.1: one second of audio takes 0.1 s to process

Below we compare Tiny, Base and Small, each on **one CPU core**, roughly like a phone or laptop.

In [ ]:
x = load("samples/cafe_noisy.wav")
sizes = {"t": "Tiny", "b": "Base", "s": "Small"}
rtfs = {}
for size, label in sizes.items():
    m = open_model(size)
    enhance_streaming(m, x[:SR])            # 先暖身一次 / warm up once
    t0 = time.perf_counter()
    enhance_streaming(m, x)
    sec = time.perf_counter() - t0
    rtfs[label] = sec / (len(x) / SR)
    print(f"{label:6s} {len(x) / SR:.1f} s 聲音 audio → {sec * 1000:6.0f} ms   RTF = {rtfs[label]:.3f}   每 per 16 ms: {rtfs[label] * 16:.2f} ms")

plt.figure(figsize=(6, 3))
plt.barh(list(rtfs), list(rtfs.values()), color=["#3cc9b3", "#1d6fb8", "#b8521a"])
plt.axvline(1.0, color="gray", ls="--"); plt.text(1.0, 2.3, " RTF = 1（即時的極限 real-time limit）", color="gray")
plt.xlabel("RTF（越小越快 lower is faster）"); plt.xlim(0, max(1.2, max(rtfs.values()) * 1.1))
plt.tight_layout(); plt.show()

## 5. 延遲：聲音晚了多少？ / Latency: how late is the output?

降噪後的聲音一定比原本的晚一點點。FastEnhancer 的輸出比輸入**晚 1 格（16 毫秒）**，
再加上收集這 1 格聲音也要 16 毫秒，所以模型本身的延遲大約是 **32 毫秒**。

人講話時，對話延遲大約在 150 毫秒以內就不太會察覺，所以 32 毫秒很夠用。
下面把**沒有對齊**的原始輸出放大來看，可以看到它整個往右移了一格。

🇬🇧 The denoised audio always comes out a little late. FastEnhancer's output is **one step (16 ms) behind** its input,
and collecting that step takes another 16 ms, so the model's own latency is about **32 ms**.

In conversation, delays under about 150 ms are hard to notice, so 32 ms is plenty fast.
Below we zoom into the **unaligned** output: it is shifted right by exactly one step.

In [ ]:
m = open_model("b")
memory = {i.name: np.zeros(i.shape, np.float32) for i in m.get_inputs() if i.name.startswith("cache_in")}
raw_out = []
xc = load("samples/cafe_clean.wav")
for start in range(0, len(xc) - HOP, HOP):
    r = m.run(None, {"wav_in": xc[None, start:start + HOP], **memory})
    raw_out.append(r[0][0])
    for j, name in enumerate(memory):
        memory[name] = r[j + 1]
raw_out = np.concatenate(raw_out)

# 自動挑一段最大聲的母音來畫 / pick the loudest vowel to plot
energy = [np.sum(xc[i:i + 3 * HOP] ** 2) for i in range(0, len(xc) - 4 * HOP, HOP)]
s0 = int(np.argmax(energy)) * HOP
seg = slice(s0, s0 + 3 * HOP)
t = np.arange(seg.start, seg.stop) / SR * 1000
plt.figure(figsize=(10, 3))
plt.plot(t, xc[seg], label="輸入 Input")
plt.plot(t, raw_out[seg], label="輸出（未對齊）Output (unaligned)", alpha=0.8)
plt.xlabel("時間 Time (ms)"); plt.legend(); plt.title("輸出比輸入晚了 256 點 = 16 毫秒 / Output lags input by 256 samples = 16 ms")
plt.tight_layout(); plt.show()

lag = np.argmax(np.correlate(raw_out[:SR * 3], xc[:SR * 3 - 1024], "valid")[:1024])
print(f"量到的延遲 Measured delay: {lag} 個取樣點 samples = {lag / SR * 1000:.0f} ms")

## 6. 實驗：拿掉記憶會怎樣？ / Experiment: what if we remove the memory?

如果每一格都讓模型「失憶」（`keep_memory=False`，記憶永遠是 0），會發生什麼事？
先猜猜看，再執行。

🇬🇧 What happens if the model "forgets" at every step (`keep_memory=False`, the memory is always zero)?
Make a guess first, then run the cell.

In [ ]:
x = load("samples/cafe_noisy.wav")
with_mem = enhance_streaming(model, x, keep_memory=True)
no_mem = enhance_streaming(model, x, keep_memory=False)

fig, axes = plt.subplots(2, 1, figsize=(10, 4.5), sharex=True)
spec(axes[0], with_mem, "有記憶（正常）With memory (normal)")
spec(axes[1], no_mem, "沒有記憶（每 16 毫秒失憶一次）No memory (forgets every 16 ms)")
plt.tight_layout(); plt.show()
print("有記憶 With memory:"); display(Audio(with_mem, rate=SR))
print("沒有記憶 No memory:"); display(Audio(no_mem, rate=SR))

**為什麼？** 只看 16 毫秒，模型分不出這是人聲還是噪音，也拼不回完整的波形。
記憶讓模型知道「剛剛一直都有這個嗡嗡聲，那它應該是噪音」。
串流模型的關鍵設計，就是**用很少的記憶記住過去**，而不是每次都重看整段聲音。

🇬🇧 **Why?** From 16 ms alone, the model cannot tell speech from noise, nor stitch the waveform back together.
Memory lets it reason: "this hum has been here all along, so it must be noise."
The key design of a streaming model is to **remember the past with a small memory**, instead of re-reading the whole recording every time.

## 7. 實驗：噪音越大，模型還撐得住嗎？ / Experiment: how much noise can it handle?

我們手上有乾淨答案，所以可以自己調整噪音大小，並用 **SI-SDR** 打分數：
它衡量輸出和乾淨答案有多接近，單位是 dB，越高越好。

🇬🇧 Since we have the clean reference, we can turn the noise up or down and score the result with **SI-SDR**:
it measures how close the output is to the clean speech, in dB; higher is better.

In [ ]:
def si_sdr(est, ref):
    est, ref = est - est.mean(), ref - ref.mean()
    proj = (est @ ref) / (ref @ ref) * ref
    return 10 * np.log10((proj @ proj) / ((est - proj) @ (est - proj)))

clean = load("samples/cafe_clean.wav")
noise = load("samples/cafe_noisy.wav") - clean          # 噪音 = 有噪音 - 乾淨 / noise = noisy - clean (the example is 0 dB)
snrs = [-5, 0, 5, 10, 20]
before, after = [], []
for snr in snrs:
    mix = clean + noise * 10 ** (-snr / 20)
    mix = mix / np.abs(mix).max() * 0.9
    out = enhance_streaming(model, mix.astype(np.float32))
    before.append(si_sdr(mix, clean)); after.append(si_sdr(out, clean))
    print(f"SNR {snr:+3d} dB：降噪前 before {before[-1]:5.1f} dB → 降噪後 after {after[-1]:5.1f} dB")

plt.figure(figsize=(6, 3.5))
plt.plot(snrs, before, "o-", label="降噪前 Before")
plt.plot(snrs, after, "o-", label="降噪後 After")
plt.xlabel("噪音大小 SNR (dB)，越左越吵 / noisier to the left"); plt.ylabel("SI-SDR (dB)，越高越好 / higher is better")
plt.legend(); plt.grid(alpha=0.3); plt.tight_layout(); plt.show()

loudest = clean + noise * 10 ** (5 / 20)
loudest = (loudest / np.abs(loudest).max() * 0.9).astype(np.float32)
print("SNR −5 dB（噪音比人聲大聲 noise louder than speech）降噪前、後 before / after:")
display(Audio(loudest, rate=SR)); display(Audio(enhance_streaming(model, loudest), rate=SR))

## 8. 錄你自己的聲音 / Record your own voice

執行下面的 cell，瀏覽器會詢問麥克風權限，接著錄 5 秒。
可以邊講話邊敲桌子、搓塑膠袋、開風扇，看看哪些噪音消得掉。

🇬🇧 Run the cell below; your browser will ask for microphone permission, then record 5 seconds.
Talk while tapping the desk, crinkling a plastic bag or running a fan, and see which noises disappear.

In [ ]:
from IPython.display import Javascript
from base64 import b64decode
import io, subprocess

try:
    from google.colab import output
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

RECORD_JS = '''
async function record(sec) {
  const stream = await navigator.mediaDevices.getUserMedia({audio: {noiseSuppression: false, echoCancellation: false, autoGainControl: false}});
  const rec = new MediaRecorder(stream); const chunks = [];
  rec.ondataavailable = e => chunks.push(e.data);
  rec.start(); await new Promise(r => setTimeout(r, sec * 1000)); rec.stop();
  await new Promise(r => rec.onstop = r); stream.getTracks().forEach(t => t.stop());
  const buf = await new Blob(chunks).arrayBuffer();
  let s = ''; new Uint8Array(buf).forEach(b => s += String.fromCharCode(b));
  return btoa(s);
}
'''

def record(sec=5):
    display(Javascript(RECORD_JS))
    print(f"錄音中，請講話 {sec} 秒… / Recording, please speak for {sec} s…")
    data = b64decode(output.eval_js(f"record({sec})"))
    open("my_voice.webm", "wb").write(data)
    subprocess.run(["ffmpeg", "-y", "-loglevel", "error", "-i", "my_voice.webm", "-ac", "1", "-ar", str(SR), "my_voice.wav"], check=True)
    return load("my_voice.wav")

if IN_COLAB:
    mine = record(5)
    mine_enh = enhance_streaming(model, mine)
    fig, axes = plt.subplots(2, 1, figsize=(10, 4.5), sharex=True)
    spec(axes[0], mine, "我的錄音 My recording"); spec(axes[1], mine_enh, "降噪後 Denoised")
    plt.tight_layout(); plt.show()
    print("原始 Original:"); display(Audio(mine, rate=SR))
    print("降噪後 Denoised:"); display(Audio(mine_enh, rate=SR))
else:
    print("這個 cell 需要在 Colab 執行 / This cell only works in Colab")

## 延伸 / Going further

- **降噪網頁版 Web version**：https://benevolent-pithivier-3ffb14.netlify.app
  在手機上即時降噪，模型直接在手機裡跑 / live denoising on your phone, the model runs on the phone itself
- **FastEnhancer 原始碼與論文 Code and paper**：<https://github.com/aask1357/fastenhancer>，[arXiv:2509.21867](https://arxiv.org/abs/2509.21867)
- **自己架一份降噪網頁版 Host your own copy**：[課程 GitHub repo / course repo](https://github.com/tehan0530/CYCU-fastenhancer-course) 的 README「自己架一份」有三種方式 / the README lists three ways
- **想自己訓練模型？ Want to train your own?** 官方文件 / official docs：<https://aask1357.github.io/fastenhancer/>

範例音檔 Example clips：VCTK / VoiceBank 語音 speech（CC BY 4.0）＋ DEMAND 環境噪音 noise（CC BY-SA 3.0）。